<a id="top"></a>

# Lab L0806: write the description, design the schema

```yaml
title: "Lab L0806: write the description, design the schema"
keywords: tool description, rich description, parameter schema, typed function, Literal enum, Annotated, per-field description, required, bind_tools, validate arguments, lab
estimated duration: 30
```

> **Lesson:** L08. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L08/objectives.md). Solutions: `L0806_lab_solutions.ipynb`. Pairs with the [L0805 demo](L0805_lecture.ipynb).
>
> **Pure Python — no API key needed.** L08 is about *design judgment*, and you can practice that offline. In this course a tool **is** a typed Python function: `bind_tools` infers the JSON definition (name, description, parameter schema) from it. So "designing the schema" means designing the function's **signature** (types, required-ness, `Literal` enums) and its **docstring** — and then inspecting the schema LangChain infers.
>
> **After this lab you can:** write a description *for the model*, design a tight tool by typing the function well (`Annotated` descriptions, a `Literal` enum, required params), and validate sample arguments against the inferred schema — all in pure Python.

## Contents

- [1. Setup — a sparse tool to improve](#1-setup--a-sparse-tool-to-improve)
- [2. Problem 1 — Rewrite the description for the model](#2-problem-1--rewrite-the-description-for-the-model)
- [3. Problem 2 — Tighten the parameter schema](#3-problem-2--tighten-the-parameter-schema)
- [4. Problem 3 — Validate sample arguments against your schema](#4-problem-3--validate-sample-arguments-against-your-schema)
- [5. Problem 5 — Why an enum over a free string? (written)](#5-problem-5--why-an-enum-over-a-free-string-written)
- [6. Self-check](#6-self-check)

## 1. Setup — a sparse tool to improve

A `set_priority(ticket_id, level)` tool with a **sparse** docstring and a **loose** signature (`level` is any string, and even optional). Because a tool *is* a typed function here, improving the tool means improving the function. Your job across the lab is to make it a tool a model can use correctly on first read.

In [1]:
import json
from typing import Annotated, Literal

from langchain_core.utils.function_calling import convert_to_openai_tool


# A SPARSE tool: a bare docstring and a loose signature -- `level` is any string and
# even optional. In this course a tool IS a typed function; bind_tools infers the
# JSON definition printed below from it. So "improve the schema" == "improve the
# function's signature and docstring."
def set_priority(ticket_id: str, level: str = "low") -> str:
    """Sets priority."""  # sparse: no when / what / return guidance
    return f"{ticket_id} -> {level}"


def inferred_schema(fn: object) -> dict[str, object]:
    """The JSON tool definition LangChain infers from a function (name, description, params)."""
    return convert_to_openai_tool(fn)["function"]


print("sparse definition bind_tools would send:")
print(json.dumps(inferred_schema(set_priority), indent=2))

sparse definition bind_tools would send:
{
  "name": "set_priority",
  "description": "Sets priority.",
  "parameters": {
    "properties": {
      "ticket_id": {
        "type": "string"
      },
      "level": {
        "default": "low",
        "type": "string"
      }
    },
    "required": [
      "ticket_id"
    ],
    "type": "object"
  }
}


[↑ Back to top](#top)

## 2. Problem 1 — Rewrite the description for the model

Write `RICH_DESCRIPTION`: a 3–5 sentence string that answers all four questions from the lecture — *what it does*, *when to call it*, *when NOT to call it*, *what it returns*. Include the accepted `level` values and a one-line example. This becomes the function's **docstring** in Problem 2 — the description the model actually sees. The check below asserts your description mentions the key facts; the *wording* is yours.

In [2]:
RICH_DESCRIPTION = (
    "Set the priority of an existing support ticket. Call this when the user explicitly "
    "asks to change a ticket's priority and names (or has named) the ticket. Do NOT call it "
    "to create a ticket or to read its current priority — it only updates an existing one. "
    "The level must be one of 'low', 'medium', or 'high'. Returns "
    "{'ticket_id', 'level', 'updated': true} on success, e.g. "
    "{'ticket_id': 'T-42', 'level': 'high', 'updated': true}."
)

# Lightweight check that the description covers the essentials (not graded on wording):
low = RICH_DESCRIPTION.lower()
for needle in ["ticket", "low", "high", "return"]:
    print(f"mentions {needle!r}:", needle in low)
assert len(RICH_DESCRIPTION.split()) >= 25, "aim for 3-5 real sentences"

mentions 'ticket': True
mentions 'low': True
mentions 'high': True
mentions 'return': True


[↑ Back to top](#top)

## 3. Problem 2 — Type the function (and read its inferred schema)

Design the tight tool by **typing the function**: write `set_priority` with `ticket_id` and `level` both **required** (no defaults); `level` a `Literal["low", "medium", "high"]` (that becomes an **enum**); and each parameter `Annotated[type, "..."]` with a per-field description carrying an example or a constraint. Reuse your `RICH_DESCRIPTION` as the docstring. Then read the schema LangChain infers — `TIGHT_SCHEMA` is exactly the parameter block `bind_tools` would send.

In [3]:
def set_priority(
    ticket_id: Annotated[str, "the ticket id to update, e.g. 'T-42'."],
    level: Annotated[
        Literal["low", "medium", "high"], "the new priority; one of low, medium, high."
    ],
) -> str:
    """Placeholder; the real description is set from RICH_DESCRIPTION just below."""
    return f"{ticket_id} -> {level}"


# The docstring IS the description the model sees -- reuse the rich one from Problem 1.
set_priority.__doc__ = RICH_DESCRIPTION

# TIGHT_SCHEMA is the parameter block bind_tools infers: required, enum, per-field descriptions.
TIGHT_SCHEMA: dict[str, object] = inferred_schema(set_priority)["parameters"]
print(json.dumps(TIGHT_SCHEMA, indent=2))

{
  "properties": {
    "ticket_id": {
      "description": "the ticket id to update, e.g. 'T-42'.",
      "type": "string"
    },
    "level": {
      "description": "the new priority; one of low, medium, high.",
      "enum": [
        "low",
        "medium",
        "high"
      ],
      "type": "string"
    }
  },
  "required": [
    "ticket_id",
    "level"
  ],
  "type": "object"
}


[↑ Back to top](#top)

## 4. Problem 3 — Validate sample arguments against your schema

Implement `validate(args, schema)` (a small subset of JSON-Schema validation): every `required` key present; each present value matches its property `type` (`string`->`str`, `integer`->`int`); and if a property has an `enum`, the value is in it. Return `"ok"` or a short reason string. Run it over the sample calls.

In [4]:
SAMPLES = [
    {"ticket_id": "T-42", "level": "high"},  # ok
    {"ticket_id": "T-7"},  # missing required level
    {"ticket_id": "T-9", "level": "urgent"},  # not in enum
    {"ticket_id": 9, "level": "low"},  # wrong type for ticket_id
]


TYPES = {"string": str, "integer": int, "boolean": bool}


def validate(args: dict[str, object], schema: dict[str, object]) -> str:
    """Tiny JSON-Schema check: required present, types match, enum membership."""
    props = schema["properties"]
    for key in schema.get("required", []):
        if key not in args:
            return f"missing required field {key!r}"
    for key, value in args.items():
        spec = props.get(key)
        if spec is None:
            return f"unknown field {key!r}"
        expected = TYPES[spec["type"]]
        # bool is a subclass of int in Python; guard it out for integer fields.
        if not isinstance(value, expected) or (expected is int and isinstance(value, bool)):
            return f"field {key!r} must be {spec['type']}, got {type(value).__name__}"
        if "enum" in spec and value not in spec["enum"]:
            return f"field {key!r} must be one of {spec['enum']}, got {value!r}"
    return "ok"


for s in SAMPLES:
    print(s, "->", validate(s, TIGHT_SCHEMA))

{'ticket_id': 'T-42', 'level': 'high'} -> ok
{'ticket_id': 'T-7'} -> missing required field 'level'
{'ticket_id': 'T-9', 'level': 'urgent'} -> field 'level' must be one of ['low', 'medium', 'high'], got 'urgent'
{'ticket_id': 9, 'level': 'low'} -> field 'ticket_id' must be string, got int


[↑ Back to top](#top)

## 5. Problem 5 — Why an enum over a free string? (written)

Your tight tool types `level` as a `Literal` enum instead of a free-form string. In a sentence or two: what does the enum buy you in terms of *model behavior* and *validation* (think about the L0805 demo's sparse-vs-rich contrast)?

_Write your answer by editing this cell (double-click)._

An enum turns an open-ended field into a closed set of legal values. For **model behavior**: the model sees the exact allowed values in the schema, so it stops inventing synonyms ("urgent", "P1") and picks one of `low`/`medium`/`high` — the same sparse-vs-rich effect from L0805, but enforced by the type instead of hoped for in prose. For **validation**: any out-of-set value becomes a mechanical, catchable error (`"urgent" not in [...]`) that your code can reject with a precise message the model can correct next turn, rather than a bad string silently flowing into the tool.

[↑ Back to top](#top)

## 6. Self-check

Compare against `L0806_lab_solutions.ipynb`. Done when your `RICH_DESCRIPTION` answers all four questions, your typed `set_priority` makes both fields required with a `Literal` enum on `level` (so `TIGHT_SCHEMA` shows `required` + `enum` + per-field descriptions), and `validate` returns `ok` for the first sample and a useful reason for the other three.

[↑ Back to top](#top)